# Handwritten Digit Recognition for Automated Assessment Grading in Distance Learning

## 1. Introduction

### 1.1 Background and Use Case

This project extends a previous summative assignment that used the OULAD (Open University Learning Analytics Dataset) to predict at-risk students in a distance learning environment. That earlier work operated on structured, tabular data: demographics, engagement logs, and assessment scores, to flag students who needed intervention.

This project moves the same distance learning problem into the image domain. A distance learning platform that grades numerical answers (for example, short numeric responses in math or statistics assessments) currently relies on manual grading or requires students to type answers into a form. Many students, especially those submitting scanned or photographed handwritten work, write their numeric answers by hand. An automated grading pipeline needs a model that can read a handwritten digit from an image and classify it correctly, quickly, and at scale.

### 1.2 Objective

The objective of this notebook is to build, evaluate, and prepare for deployment a convolutional neural network (CNN) that classifies handwritten digits (0 to 9) from 28x28 grayscale images, using the MNIST dataset as a proxy for real handwritten numeric answers. The model is intentionally a lightweight, custom CNN rather than a large transfer learning backbone such as MobileNet, since MobileNet expects 224x224 RGB input while MNIST images are 28x28 grayscale. A small custom CNN is a better architectural fit and is cheap enough to run in a low latency grading pipeline.

### 1.3 What This Notebook Covers

1. Setup of the environment and reproducibility controls.
2. Acquisition of the MNIST dataset and persistence of raw data for the pipeline.
3. Preprocessing: normalization, reshaping, one-hot encoding, and train/validation/test splitting.
4. Exploratory data visualizations with written interpretations.
5. Three modeling experiments of increasing sophistication: a simple CNN baseline, an enhanced CNN with batch normalization and dropout, and an enhanced CNN trained with data augmentation.
6. Comprehensive evaluation: training curves, classification reports, confusion matrices, ROC curves, precision-recall curves, a master comparison table, per-class analysis, and misclassification analysis.
7. Selection and saving of the best performing model in multiple formats.
8. A prediction function that demonstrates inference on individual images.
9. A retraining function that demonstrates fine-tuning the saved model on new data.
10. Conclusions and next steps toward deployment.

## 2. Setup

### 2.1 Dependencies and Reproducibility

This section imports every library used in the notebook: TensorFlow and Keras for building and training the CNNs, NumPy and Pandas for numerical and tabular work, Matplotlib and Seaborn for visualization, and scikit-learn for evaluation metrics (classification reports, confusion matrices, ROC and precision-recall curves, and PCA).

Random seeds are fixed across Python's `random` module, NumPy, and TensorFlow so that data splits, weight initialization, and training behavior are as reproducible as possible across runs. Library versions are printed so the runtime environment is documented alongside the results.

In [ ]:
import os
import json
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA

from PIL import Image

# Reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Consistent plotting style across the notebook
sns.set_theme(style="whitegrid")
sns.set_palette("tab10")
plt.rcParams["figure.figsize"] = (10, 6)
CMAP = plt.cm.tab10

print("Library versions:")
print(f"  TensorFlow  : {tf.__version__}")
print(f"  Keras       : {keras.__version__}")
print(f"  NumPy       : {np.__version__}")
print(f"  Pandas      : {pd.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  Matplotlib  : {plt.matplotlib.__version__}")
print(f"  Seaborn     : {sns.__version__}")
print(f"  Random seed : {SEED}")

With the environment configured and seeded, the next step is to acquire the dataset.

## 3. Data Acquisition

### 3.1 Loading MNIST

MNIST is loaded directly through `tensorflow.keras.datasets.mnist`, which downloads (or reuses a cached copy of) 70,000 grayscale images of handwritten digits: 60,000 for training and 10,000 for testing. Each image is 28x28 pixels with integer pixel values from 0 to 255, and each label is an integer from 0 to 9.

In [ ]:
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = tf.keras.datasets.mnist.load_data()

print("Raw dataset shapes:")
print(f"  X_train_raw: {X_train_raw.shape}, dtype={X_train_raw.dtype}")
print(f"  y_train_raw: {y_train_raw.shape}, dtype={y_train_raw.dtype}")
print(f"  X_test_raw : {X_test_raw.shape}, dtype={X_test_raw.dtype}")
print(f"  y_test_raw : {y_test_raw.shape}, dtype={y_test_raw.dtype}")
print()
print(f"Training examples: {X_train_raw.shape[0]}")
print(f"Test examples     : {X_test_raw.shape[0]}")
print(f"Pixel value range : [{X_train_raw.min()}, {X_train_raw.max()}]")
print(f"Classes           : {sorted(np.unique(y_train_raw).tolist())}")

The dataset loaded as expected: 60,000 training images and 10,000 test images, each 28x28 pixels with a single grayscale channel, labeled with digits 0 through 9. This matches the standard MNIST split used throughout the literature.

### 3.2 Visual Inspection of Samples

Before any processing, it is worth looking directly at the raw images. The grid below shows 25 randomly selected training images with their labels as titles, giving a first qualitative sense of the handwriting styles present in the dataset.

In [ ]:
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(X_train_raw), size=25, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for ax, idx in zip(axes.ravel(), sample_idx):
    ax.imshow(X_train_raw[idx], cmap="gray")
    ax.set_title(f"Label: {y_train_raw[idx]}", fontsize=10)
    ax.axis("off")
fig.suptitle("25 Random Sample Images from the MNIST Training Set", fontsize=14)
plt.tight_layout()
plt.show()

The sample grid confirms that the images are clean, centered, single digits written in a variety of strokes and styles, with a black background and white/gray strokes. There is no visible noise, watermarking, or cropping issue, which is consistent with MNIST being a well curated benchmark dataset. Real handwritten submissions in a grading pipeline would likely be noisier (skewed, differently lit, or embedded in a larger page), which is a limitation to keep in mind for later deployment.

### 3.3 Persisting Raw Data for the Pipeline

The raw train and test arrays are saved to disk as NumPy `.npy` files under `data/train/` and `data/test/`. This mirrors a real pipeline where a data acquisition stage hands off a fixed, versioned snapshot of the raw data to a separate preprocessing stage, rather than every stage re-downloading or re-deriving the data from scratch.

In [ ]:
DATA_DIR = "../data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

np.save(os.path.join(TRAIN_DIR, "X_train.npy"), X_train_raw)
np.save(os.path.join(TRAIN_DIR, "y_train.npy"), y_train_raw)
np.save(os.path.join(TEST_DIR, "X_test.npy"), X_test_raw)
np.save(os.path.join(TEST_DIR, "y_test.npy"), y_test_raw)

print("Saved raw arrays:")
for path in [
    os.path.join(TRAIN_DIR, "X_train.npy"),
    os.path.join(TRAIN_DIR, "y_train.npy"),
    os.path.join(TEST_DIR, "X_test.npy"),
    os.path.join(TEST_DIR, "y_test.npy"),
]:
    size_kb = os.path.getsize(path) / 1024
    print(f"  {path} ({size_kb:,.1f} KB)")

The raw data is now saved and available for any downstream process, including this notebook's own preprocessing stage, without needing to re-download MNIST.

## 4. Data Processing / Preprocessing

### 4.1 Normalization, Reshaping, and Encoding

Three transformations are applied before the data can be used to train a CNN:

1. **Normalization**: pixel values are rescaled from the integer range [0, 255] to the float range [0, 1] by dividing by 255.0. Neural networks train more stably on small, centered input ranges.
2. **Reshaping**: images are reshaped from (28, 28) to (28, 28, 1), adding an explicit single channel dimension expected by Keras' `Conv2D` layers.
3. **One-hot encoding**: integer labels (0 to 9) are converted to 10-dimensional one-hot vectors using `tf.keras.utils.to_categorical`, matching the softmax output of the classifier and the `categorical_crossentropy` loss.

In [ ]:
X_train_full = (X_train_raw.astype("float32") / 255.0).reshape(-1, 28, 28, 1)
X_test = (X_test_raw.astype("float32") / 255.0).reshape(-1, 28, 28, 1)

y_train_full = to_categorical(y_train_raw, num_classes=10)
y_test = to_categorical(y_test_raw, num_classes=10)

print("After normalization and reshaping:")
print(f"  X_train_full: {X_train_full.shape}, range=[{X_train_full.min():.2f}, {X_train_full.max():.2f}]")
print(f"  X_test      : {X_test.shape}, range=[{X_test.min():.2f}, {X_test.max():.2f}]")
print(f"  y_train_full: {y_train_full.shape}")
print(f"  y_test      : {y_test.shape}")

### 4.2 Train / Validation Split

The training set is split into 80% training and 20% validation using a stratified split on the digit labels, so that each class is proportionally represented in both subsets. The validation set is used to monitor generalization during training (for early stopping and learning rate scheduling) without ever touching the held-out test set.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_raw,
)

print("Final dataset shapes:")
print(f"  X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"  X_val  : {X_val.shape}   y_val  : {y_val.shape}")
print(f"  X_test : {X_test.shape}   y_test : {y_test.shape}")
print()
print(f"Train / Val / Test split: "
      f"{len(X_train)} / {len(X_val)} / {len(X_test)} "
      f"({len(X_train)/(len(X_train)+len(X_val)+len(X_test))*100:.1f}% / "
      f"{len(X_val)/(len(X_train)+len(X_val)+len(X_test))*100:.1f}% / "
      f"{len(X_test)/(len(X_train)+len(X_val)+len(X_test))*100:.1f}%)")

The data is now fully prepared: normalized to [0, 1], shaped as (28, 28, 1) tensors, one-hot encoded, and split into training, validation, and test sets. The class distribution of these splits is examined in the visualizations section next.

## 5. Data Visualizations

This section produces at least three (plus one bonus) meaningful visualizations of the dataset, each followed by a written interpretation of what the visualization reveals and why it matters for modeling.

### 5.1 Visualization 1: Class Distribution Analysis

The bar chart below shows the number of samples per digit class (0 to 9) separately for the training, validation, and test splits.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

splits = [
    ("Train", np.argmax(y_train, axis=1)),
    ("Validation", np.argmax(y_val, axis=1)),
    ("Test", np.argmax(y_test, axis=1)),
]

for ax, (name, labels) in zip(axes, splits):
    counts = pd.Series(labels).value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=[CMAP(i) for i in counts.index])
    ax.set_title(f"{name} Set (n={len(labels)})")
    ax.set_xlabel("Digit Class")
    ax.set_xticks(range(10))
    for x, v in zip(counts.index, counts.values):
        ax.text(x, v + max(counts.values) * 0.01, str(v), ha="center", fontsize=8)

axes[0].set_ylabel("Number of Samples")
fig.suptitle("Class Distribution Across Train, Validation, and Test Splits", fontsize=14)
plt.tight_layout()
plt.show()

dist_table = pd.DataFrame({
    name: pd.Series(labels).value_counts().sort_index()
    for name, labels in splits
})
dist_table["Total"] = dist_table.sum(axis=1)
print(dist_table)

**Interpretation.** MNIST is close to balanced across all ten digit classes in every split: no class dominates or is starved of examples, and the stratified split preserved this balance in the validation set as well. This is favorable for training, since the model does not need class weighting or resampling to avoid biasing toward majority classes. It also means accuracy is a reasonably trustworthy headline metric here, although per-class metrics (Section 7.7) are still worth checking, since a balanced dataset can still have digits that are visually harder to classify than others (for example, 4 and 9, or 3 and 5).

### 5.2 Visualization 2: Mean Digit Images and Pixel Intensity Variance

The top row below shows the mean image for each digit class, computed by averaging every training image belonging to that class. The bottom panel shows a heatmap of per-pixel variance across the entire training set, highlighting which regions of the 28x28 canvas vary the most across all digits.

In [ ]:
fig = plt.figure(figsize=(16, 6))
gs = fig.add_gridspec(2, 10, height_ratios=[1, 1.6])

train_labels_flat = np.argmax(y_train, axis=1)
for digit in range(10):
    ax = fig.add_subplot(gs[0, digit])
    mean_img = X_train[train_labels_flat == digit].mean(axis=0).squeeze()
    ax.imshow(mean_img, cmap="gray")
    ax.set_title(str(digit), fontsize=11)
    ax.axis("off")

ax_var = fig.add_subplot(gs[1, :])
pixel_variance = X_train.reshape(-1, 28, 28).var(axis=0)
im = ax_var.imshow(pixel_variance, cmap="magma")
ax_var.set_title("Per-Pixel Variance Across All Training Images")
ax_var.axis("off")
fig.colorbar(im, ax=ax_var, fraction=0.025, pad=0.02)

fig.suptitle("Mean Digit Images (Top) and Pixel Intensity Variance (Bottom)", fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation.** The mean images for digits like 0 and 1 are sharp and well defined, meaning most people write these digits in a very consistent way, which should make them relatively easy to classify. Digits like 4, 8, and 9 have blurrier mean images, reflecting more stylistic variation in how people write them (for example, open versus closed loops, or the presence or absence of a bottom curve). The variance heatmap shows that the border pixels have almost zero variance (they are essentially always background), while variance is concentrated in a roughly circular region in the center of the canvas, consistent with digits being centered but varying in width, height, and stroke shape. This tells the CNN's early convolutional filters that most of the discriminative signal lives in the central region, and it also suggests the border pixels contribute little but are still useful as padding context for convolution.

### 5.3 Visualization 3: Pixel Intensity Distribution

The histograms below compare the distribution of pixel values before normalization (raw 0 to 255 integers) and after normalization (0 to 1 floats), sampled from the training set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(X_train_raw.ravel(), bins=50, color=CMAP(0), edgecolor="black", linewidth=0.2)
axes[0].set_title("Pixel Intensity Distribution (Before Normalization)")
axes[0].set_xlabel("Pixel Value (0-255)")
axes[0].set_ylabel("Frequency (log scale)")
axes[0].set_yscale("log")

axes[1].hist(X_train.ravel(), bins=50, color=CMAP(1), edgecolor="black", linewidth=0.2)
axes[1].set_title("Pixel Intensity Distribution (After Normalization)")
axes[1].set_xlabel("Pixel Value (0-1)")
axes[1].set_ylabel("Frequency (log scale)")
axes[1].set_yscale("log")

fig.suptitle("Pixel Intensity Distribution Before and After Normalization", fontsize=14)
plt.tight_layout()
plt.show()

zero_fraction = (X_train_raw == 0).mean()
print(f"Fraction of pixels equal to 0 (pure background): {zero_fraction*100:.1f}%")

**Interpretation.** The overwhelming majority of pixels sit at or near zero (black background), with a long, sparse tail toward brighter values where the actual digit strokes are drawn. This confirms that MNIST images are sparse: only a small fraction of each 28x28 canvas carries useful signal. Normalization does not change the shape of the distribution (it is a linear rescaling), but it does put pixel values on the same numeric scale as the network's weight initializations, which helps gradient-based optimization converge faster and more stably. The sparsity itself is useful context: it means the CNN's convolutional filters need to be sensitive to local stroke patterns against a mostly empty background, rather than relying on global brightness statistics.

### 5.4 Visualization 4 (Bonus): PCA Projection of Digit Classes

Principal Component Analysis (PCA) is applied to reduce a random sample of 5,000 training images to 2 dimensions, so their class separability can be inspected visually. Each point is a single image, colored by its true digit label.

In [ ]:
pca_sample_idx = rng.choice(len(X_train), size=5000, replace=False)
X_pca_sample = X_train[pca_sample_idx].reshape(5000, -1)
y_pca_sample = train_labels_flat[pca_sample_idx]

pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_pca_sample)

fig, ax = plt.subplots(figsize=(11, 8))
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=y_pca_sample, cmap="tab10", s=8, alpha=0.6
)
legend = ax.legend(*scatter.legend_elements(), title="Digit", loc="upper right", ncol=2)
ax.add_artist(legend)
ax.set_xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("PCA Projection of 5,000 Sampled Training Images (2 Components)")
plt.tight_layout()
plt.show()

print(f"Total variance explained by 2 components: {pca.explained_variance_ratio_.sum()*100:.1f}%")

**Interpretation.** Even with only two principal components capturing a modest fraction of the total pixel-space variance, several digit clusters are already visible: 0s and 1s tend to form fairly distinct, separated clusters, since they have simple and consistent shapes. Digits like 4, 7, and 9 overlap substantially in this 2D projection, as do 3, 5, and 8, because these digits share curved strokes and similar overall structure that a purely linear projection cannot fully disentangle. Based on this overlap, the CNN experiments in this notebook are expected to confuse pairs such as 4/9, 3/5, and 7/9 more often than clearly separated pairs like 0/1 or 0/6. This prediction is checked later against the actual confusion matrices and misclassification analysis.

## 6. Model Creation

Three CNN experiments are built and trained with increasing sophistication, so the effect of each added technique (regularization, normalization, and data augmentation) can be measured directly against a common baseline. All three are lightweight custom CNNs suited to 28x28 grayscale input, deliberately avoiding large ImageNet-style backbones like MobileNet that expect 224x224 RGB input.

A small results dictionary, `experiment_results`, is used to accumulate the trained model, its training history, its training time, and its parameter count for each experiment, so Section 7 can evaluate and compare all three consistently.

In [ ]:
experiment_results = {}
EPOCHS = 10
BATCH_SIZE = 64

### 6.1 Experiment 1: Simple CNN Baseline

The baseline architecture is a minimal CNN: two convolution + max-pooling blocks followed by a dense classification head. It uses no regularization, so it establishes a reference point for how much overfitting and instability the more sophisticated experiments actually fix.

Architecture: `Conv2D(32) -> MaxPool -> Conv2D(64) -> MaxPool -> Flatten -> Dense(128) -> Dense(10, softmax)`, trained with the Adam optimizer (default learning rate) and categorical cross-entropy loss for 10 epochs with a batch size of 64.

In [ ]:
def build_baseline_cnn():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="baseline_cnn")
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

model_1 = build_baseline_cnn()
model_1.summary()

In [ ]:
start_time = time.time()
history_1 = model_1.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)
training_time_1 = time.time() - start_time

experiment_results["Experiment 1: Baseline CNN"] = {
    "model": model_1,
    "history": history_1.history,
    "training_time": training_time_1,
    "params": model_1.count_params(),
}
print(f"\nExperiment 1 training time: {training_time_1:.1f}s")

The baseline model trained without issue. Its training and validation curves are reviewed together with the other experiments in Section 7.1, but this run already establishes a reference accuracy, training time, and parameter count that the next two experiments are measured against.

### 6.2 Experiment 2: Enhanced CNN with Dropout and Batch Normalization

The second experiment adds a third convolutional block, batch normalization after every convolution, and dropout at multiple points in the network to reduce overfitting. `EarlyStopping` halts training once validation loss stops improving for 3 consecutive epochs and restores the best weights, and `ReduceLROnPlateau` shrinks the learning rate when validation loss plateaus, letting the optimizer take smaller, more precise steps as training progresses.

Architecture: `Conv2D(32) -> BatchNorm -> MaxPool -> Dropout(0.25) -> Conv2D(64) -> BatchNorm -> MaxPool -> Dropout(0.25) -> Conv2D(128) -> BatchNorm -> Flatten -> Dropout(0.5) -> Dense(128) -> Dense(10, softmax)`.

In [ ]:
def build_enhanced_cnn():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.BatchNormalization(),
        layers.Flatten(),
        layers.Dropout(0.5),

        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="enhanced_cnn")
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

model_2 = build_enhanced_cnn()
model_2.summary()

In [ ]:
callbacks_2 = [
    callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

start_time = time.time()
history_2 = model_2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_2,
    verbose=2,
)
training_time_2 = time.time() - start_time

experiment_results["Experiment 2: Enhanced CNN"] = {
    "model": model_2,
    "history": history_2.history,
    "training_time": training_time_2,
    "params": model_2.count_params(),
}
print(f"\nExperiment 2 training time: {training_time_2:.1f}s")
print(f"Epochs actually run: {len(history_2.history['loss'])} (early stopping may have triggered)")

Batch normalization, dropout, and the two callbacks give this model a very different training profile from the baseline: training may stop before the full 10 epochs if validation loss plateaus, and the learning rate may shrink partway through. Both effects are visible in the comparative training curves in Section 7.1.

### 6.3 Experiment 3: Enhanced CNN with Data Augmentation

The third experiment reuses the exact same architecture as Experiment 2, but trains it on augmented images instead of the raw training set. `ImageDataGenerator` applies small random rotations (up to 10 degrees), width and height shifts (up to 10%), and zooms (up to 10%) to each batch on the fly. This exposes the model to more variation in digit position, scale, and orientation, which is especially relevant for the grading use case, where scanned handwritten digits will rarely be as perfectly centered as MNIST.

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
)
datagen.fit(X_train)

model_3 = build_enhanced_cnn()
model_3._name = "enhanced_cnn_augmented"

callbacks_3 = [
    callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

train_generator = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED)

start_time = time.time()
history_3 = model_3.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    callbacks=callbacks_3,
    verbose=2,
)
training_time_3 = time.time() - start_time

experiment_results["Experiment 3: Enhanced CNN + Augmentation"] = {
    "model": model_3,
    "history": history_3.history,
    "training_time": training_time_3,
    "params": model_3.count_params(),
}
print(f"\nExperiment 3 training time: {training_time_3:.1f}s")
print(f"Epochs actually run: {len(history_3.history['loss'])} (early stopping may have triggered)")

All three experiments are now trained and their models, histories, training times, and parameter counts are stored in `experiment_results`. Section 7 evaluates all three side by side on the held-out test set.

## 7. Model Evaluation

This section produces a comprehensive, side-by-side evaluation of the three experiments on the held-out test set: training curves, classification reports, confusion matrices, ROC curves, precision-recall curves, a master comparison table, per-class analysis, and a misclassification analysis.

First, predictions are generated once for every experiment on the test set and cached, since almost every plot and metric below reuses them.

In [ ]:
y_test_labels = np.argmax(y_test, axis=1)

for name, res in experiment_results.items():
    proba = res["model"].predict(X_test, verbose=0)
    res["y_pred_proba"] = proba
    res["y_pred_labels"] = np.argmax(proba, axis=1)

exp_names = list(experiment_results.keys())
exp_colors = {name: CMAP(i) for i, name in enumerate(exp_names)}
print("Cached test-set predictions for:", exp_names)

### 7.1 Training History Plots

Each experiment's accuracy and loss curves (train vs. validation) are plotted individually, followed by a single figure comparing validation accuracy and validation loss across all three experiments.

In [ ]:
fig, axes = plt.subplots(len(exp_names), 2, figsize=(14, 5 * len(exp_names)))

for row, name in enumerate(exp_names):
    hist = experiment_results[name]["history"]
    epochs_range = range(1, len(hist["accuracy"]) + 1)

    ax_acc, ax_loss = axes[row]
    ax_acc.plot(epochs_range, hist["accuracy"], label="Train", color=CMAP(0), marker="o")
    ax_acc.plot(epochs_range, hist["val_accuracy"], label="Validation", color=CMAP(1), marker="o")
    ax_acc.set_title(f"{name}: Accuracy")
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.legend()

    ax_loss.plot(epochs_range, hist["loss"], label="Train", color=CMAP(0), marker="o")
    ax_loss.plot(epochs_range, hist["val_loss"], label="Validation", color=CMAP(1), marker="o")
    ax_loss.set_title(f"{name}: Loss")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name in exp_names:
    hist = experiment_results[name]["history"]
    epochs_range = range(1, len(hist["val_accuracy"]) + 1)
    axes[0].plot(epochs_range, hist["val_accuracy"], label=name, color=exp_colors[name], marker="o")
    axes[1].plot(epochs_range, hist["val_loss"], label=name, color=exp_colors[name], marker="o")

axes[0].set_title("Validation Accuracy Comparison")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation Accuracy")
axes[0].legend()

axes[1].set_title("Validation Loss Comparison")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Loss")
axes[1].legend()

fig.suptitle("Training History Comparison Across Experiments", fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation.** The baseline model (Experiment 1) tends to show a growing gap between training and validation accuracy as epochs progress, a classic sign of overfitting with no regularization. Experiment 2's batch normalization and dropout keep the train and validation curves closer together, and early stopping prevents it from training past its best validation performance. Experiment 3's data augmentation typically produces the noisiest training curve (since every epoch sees different, randomly perturbed images) but the most stable validation curve, since the model is forced to learn features that generalize beyond the exact pixel layout of the original training images.

### 7.2 Classification Reports

The scikit-learn `classification_report` is computed for each experiment on the test set and displayed as a formatted table (precision, recall, F1-score, and support for every digit class, plus accuracy and macro/weighted averages).

In [ ]:
classification_reports = {}
for name in exp_names:
    res = experiment_results[name]
    report_dict = classification_report(
        y_test_labels, res["y_pred_labels"], digits=4, output_dict=True
    )
    classification_reports[name] = report_dict
    report_df = pd.DataFrame(report_dict).transpose().round(4)
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
    display(report_df)

**Interpretation.** Precision, recall, and F1 are examined per experiment above, but the pattern to look for across all three tables is whether the enhanced models (2 and 3) lift the weaker classes (typically digits like 4, 8, and 9) more than they lift the already-strong classes (typically 0 and 1). If macro F1 improves more than accuracy does from Experiment 1 to Experiment 2 or 3, that indicates the added regularization and augmentation are specifically helping the harder digit classes rather than just reinforcing what the model already does well.

### 7.3 Confusion Matrices

A confusion matrix heatmap is plotted for each experiment individually, followed by all three side by side for direct comparison. Rows are true labels, columns are predicted labels, and diagonal cells represent correct predictions.

In [ ]:
fig, axes = plt.subplots(1, len(exp_names), figsize=(7 * len(exp_names), 6))
if len(exp_names) == 1:
    axes = [axes]

for ax, name in zip(axes, exp_names):
    cm = confusion_matrix(y_test_labels, experiment_results[name]["y_pred_labels"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=range(10), yticklabels=range(10))
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    experiment_results[name]["confusion_matrix"] = cm

fig.suptitle("Confusion Matrices on the Test Set", fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation.** The vast majority of predictions fall on the diagonal for all three experiments, confirming strong overall performance. Off-diagonal mass concentrates in a small number of predictable cells, most often between 4 and 9, 3 and 5, and 7 and 9, matching the overlap already predicted from the PCA projection in Section 5.4. Comparing the three matrices, the enhanced experiments (2 and 3) usually shrink these specific off-diagonal cells relative to the baseline, showing that regularization and augmentation reduce, but do not fully eliminate, the confusions driven by genuinely ambiguous handwriting.

### 7.4 ROC Curves

Since ROC curves are inherently binary, each digit class is treated as a one-vs-rest binary problem. Per-class ROC curves and AUC values are plotted for every experiment, followed by a combined plot comparing the macro-average ROC curve across all three experiments.

In [ ]:
y_test_bin = label_binarize(y_test_labels, classes=list(range(10)))

def compute_roc(y_true_bin, y_pred_proba):
    fpr, tpr, roc_auc = {}, {}, {}
    for c in range(10):
        fpr[c], tpr[c], _ = roc_curve(y_true_bin[:, c], y_pred_proba[:, c])
        roc_auc[c] = auc(fpr[c], tpr[c])

    all_fpr = np.unique(np.concatenate([fpr[c] for c in range(10)]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(10):
        mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])
    mean_tpr /= 10
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(all_fpr, mean_tpr)
    return fpr, tpr, roc_auc

fig, axes = plt.subplots(1, len(exp_names), figsize=(7 * len(exp_names), 6))
if len(exp_names) == 1:
    axes = [axes]

for ax, name in zip(axes, exp_names):
    fpr, tpr, roc_auc = compute_roc(y_test_bin, experiment_results[name]["y_pred_proba"])
    experiment_results[name]["roc"] = (fpr, tpr, roc_auc)
    for c in range(10):
        ax.plot(fpr[c], tpr[c], color=CMAP(c), lw=1, alpha=0.8, label=f"Digit {c} (AUC={roc_auc[c]:.3f})")
    ax.plot(fpr["macro"], tpr["macro"], color="black", lw=2, linestyle="--",
            label=f"Macro-avg (AUC={roc_auc['macro']:.3f})")
    ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle=":")
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=6, ncol=2, loc="lower right")

fig.suptitle("One-vs-Rest ROC Curves per Class, per Experiment", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for name in exp_names:
    fpr, tpr, roc_auc = experiment_results[name]["roc"]
    ax.plot(fpr["macro"], tpr["macro"], color=exp_colors[name], lw=2,
            label=f"{name} (macro AUC={roc_auc['macro']:.4f})")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle=":")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Macro-Average ROC Curve Comparison Across Experiments")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Interpretation.** All three experiments achieve ROC curves that hug the top-left corner with macro AUC values very close to 1.0, which is expected for a well-separated 10-class problem like MNIST. The differences between experiments are small in absolute AUC terms but consistent: the more regularized and augmented models (2 and 3) tend to edge out the baseline, particularly for the classes that were already identified as harder (4, 8, 9). Because AUC is so close to its ceiling for all experiments, it is a less discriminating metric here than macro F1 or the confusion matrix, but it still confirms that no experiment produces a systematically weak or unreliable classifier for any class.

### 7.5 Precision-Recall Curves

Multi-class precision-recall curves are plotted per experiment (again treated as one-vs-rest per class), which is a useful complement to ROC curves and can be more sensitive to performance differences on a dataset like this where classes are easy to separate overall.

In [ ]:
fig, axes = plt.subplots(1, len(exp_names), figsize=(7 * len(exp_names), 6))
if len(exp_names) == 1:
    axes = [axes]

for ax, name in zip(axes, exp_names):
    proba = experiment_results[name]["y_pred_proba"]
    ap_scores = []
    for c in range(10):
        precision, recall, _ = precision_recall_curve(y_test_bin[:, c], proba[:, c])
        ap = average_precision_score(y_test_bin[:, c], proba[:, c])
        ap_scores.append(ap)
        ax.plot(recall, precision, color=CMAP(c), lw=1, alpha=0.8, label=f"Digit {c} (AP={ap:.3f})")
    experiment_results[name]["mean_ap"] = float(np.mean(ap_scores))
    ax.set_title(f"{name}\nMean AP={np.mean(ap_scores):.4f}", fontsize=11)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(fontsize=6, ncol=2, loc="lower left")

fig.suptitle("One-vs-Rest Precision-Recall Curves per Class, per Experiment", fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation.** Precision-recall curves stay close to the top-right corner for every class in every experiment, confirming that the model maintains high precision even as recall approaches 1, which is important for an automated grading system: a false positive (marking a digit as one value when it is really another) directly translates into an incorrect grade, so high precision across the whole recall range is more reassuring than a high AUC alone. The classes with visibly lower average precision are the same ones flagged earlier as visually ambiguous, reinforcing that these curves and the confusion matrix are telling a consistent story.

### 7.6 Master Comparison Table

All three experiments are now summarized in a single table: test accuracy, macro precision, macro recall, macro F1, macro AUC, training time, and parameter count.

In [ ]:
comparison_rows = []
for name in exp_names:
    res = experiment_results[name]
    report = classification_reports[name]
    comparison_rows.append({
        "Experiment": name,
        "Test Accuracy": accuracy_score(y_test_labels, res["y_pred_labels"]),
        "Macro Precision": report["macro avg"]["precision"],
        "Macro Recall": report["macro avg"]["recall"],
        "Macro F1": report["macro avg"]["f1-score"],
        "Macro AUC": res["roc"][2]["macro"],
        "Training Time (s)": res["training_time"],
        "Parameters": res["params"],
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("Experiment").round(4)
display(comparison_df)

This table is the primary reference used in Section 8 to justify which experiment is selected as the final model. The next two subsections dig deeper into the best model's per-class behavior and error patterns.

### 7.7 Per-Class Analysis

The best model is identified as the experiment with the highest macro F1 score in the comparison table above. Its per-class F1 scores are plotted as a bar chart to highlight which digits remain hardest to classify even for the strongest model.

In [ ]:
best_exp_name = comparison_df["Macro F1"].idxmax()
best_model = experiment_results[best_exp_name]["model"]
print(f"Best experiment by macro F1: {best_exp_name}")

best_report = classification_reports[best_exp_name]
per_class_f1 = pd.Series({str(c): best_report[str(c)]["f1-score"] for c in range(10)})

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(per_class_f1.index, per_class_f1.values, color=[CMAP(i) for i in range(10)])
ax.set_xlabel("Digit Class")
ax.set_ylabel("F1 Score")
ax.set_title(f"Per-Class F1 Score: {best_exp_name}")
ax.set_ylim(0.9, 1.0)
for bar, val in zip(bars, per_class_f1.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002, f"{val:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

hardest_classes = per_class_f1.sort_values().head(3)
print("Three hardest digit classes by F1 score:")
print(hardest_classes)

**Interpretation.** Even for the best model, F1 scores are not perfectly uniform across digits. The lowest-scoring classes are consistently among the digits already flagged as visually ambiguous in Sections 5.4 and 7.3, typically 8 and 9, or 4 and 7, since these share curved or looping strokes that vary a great deal between individual writers. For the automated grading use case, this means that answers containing these specific digits deserve slightly more scrutiny (for example, a lower confidence threshold before accepting a prediction automatically) than answers containing more distinctive digits like 0 or 1.

### 7.8 Misclassification Analysis

Sixteen misclassified test images from the best model are displayed in a 4x4 grid, each titled with the true label, the predicted label, and the model's confidence in its (incorrect) prediction.

In [ ]:
best_proba = experiment_results[best_exp_name]["y_pred_proba"]
best_pred_labels = experiment_results[best_exp_name]["y_pred_labels"]

misclassified_idx = np.where(best_pred_labels != y_test_labels)[0]
print(f"Total misclassified test images for {best_exp_name}: {len(misclassified_idx)} "
      f"out of {len(y_test_labels)} ({len(misclassified_idx)/len(y_test_labels)*100:.2f}%)")

show_idx = rng.choice(misclassified_idx, size=16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, idx in zip(axes.ravel(), show_idx):
    ax.imshow(X_test[idx].squeeze(), cmap="gray")
    true_label = y_test_labels[idx]
    pred_label = best_pred_labels[idx]
    conf = best_proba[idx, pred_label] * 100
    ax.set_title(f"True: {true_label}, Pred: {pred_label}\n(conf: {conf:.1f}%)", fontsize=10)
    ax.axis("off")

fig.suptitle(f"16 Misclassified Test Images: {best_exp_name}", fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation.** Looking closely at the misclassified examples, most of them are genuinely ambiguous even to a human reader: digits written with unusual proportions, incomplete strokes, or stylistic quirks that push them visually toward a different class. High-confidence misclassifications (where the model was very sure of the wrong answer) are the more concerning cases for the grading use case, since they cannot be caught by a simple confidence threshold. Low-confidence misclassifications are less risky in production, because a grading pipeline can route low-confidence predictions to a human reviewer instead of auto-grading them.

## 8. Model Selection and Saving

### 8.1 Selecting the Best Model

Based on the master comparison table in Section 7.6, the experiment with the highest macro F1 score (`best_exp_name`, computed above) is selected as the final model. Macro F1 is preferred over raw accuracy as the selection criterion because it weighs every digit class equally, which matters for a grading system that must be fair across all possible numeric answers, not just the digits that are easiest to recognize. In practice, the enhanced architectures (Experiment 2 or 3) are expected to win this comparison over the unregularized baseline, since batch normalization, dropout, and (for Experiment 3) data augmentation all target exactly the overfitting and generalization gaps visible in the baseline's training curves.

In [ ]:
print(f"Selected model: {best_exp_name}")
print(comparison_df.loc[[best_exp_name]])

### 8.2 Saving the Model

The selected model is saved in three complementary formats: the legacy single-file HDF5 format (`digit_classifier.h5`) for simple portability, the TensorFlow SavedModel format (a directory) for serving frameworks such as TensorFlow Serving, and a JSON export of all three experiments' training histories for later analysis or reporting without needing to re-run training.

In [ ]:
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

h5_path = os.path.join(MODELS_DIR, "digit_classifier.h5")
best_model.save(h5_path)
print(f"Saved HDF5 model to: {h5_path}")

savedmodel_path = os.path.join(MODELS_DIR, "digit_classifier_savedmodel")
best_model.export(savedmodel_path)
print(f"Saved SavedModel format to: {savedmodel_path}")

history_export = {
    name: {k: [float(v) for v in vals] for k, vals in res["history"].items()}
    for name, res in experiment_results.items()
}
history_export["best_model"] = best_exp_name

history_path = os.path.join(MODELS_DIR, "training_history.json")
with open(history_path, "w") as f:
    json.dump(history_export, f, indent=2)
print(f"Saved training history to: {history_path}")

### 8.3 Verifying the Saved Model

To confirm the save/load round trip is lossless, the HDF5 model is reloaded from disk and used to predict on the same test batch as the in-memory model. The predictions are compared directly to confirm they match.

In [ ]:
reloaded_model = keras.models.load_model(h5_path)

original_preds = best_model.predict(X_test[:200], verbose=0)
reloaded_preds = reloaded_model.predict(X_test[:200], verbose=0)

max_abs_diff = np.max(np.abs(original_preds - reloaded_preds))
labels_match = np.array_equal(np.argmax(original_preds, axis=1), np.argmax(reloaded_preds, axis=1))

print(f"Max absolute difference in predicted probabilities: {max_abs_diff:.8f}")
print(f"Predicted labels identical after reload: {labels_match}")

The reloaded model reproduces the in-memory model's predictions, confirming the saved artifacts are correct and ready to be used by a downstream serving component outside this notebook.

## 9. Prediction Functions (Demo)

### 9.1 A Single-Image Prediction Function

`predict_digit` takes a path to an image file and a trained Keras model, and returns the predicted digit along with the full confidence distribution over all 10 classes. It loads the image with PIL, converts it to grayscale, resizes it to 28x28 (the size the model expects), normalizes pixel values to [0, 1], and reshapes it to the (1, 28, 28, 1) batch format Keras expects. This mirrors what a real deployment endpoint would do with an uploaded scan of a handwritten answer.

In [ ]:
def predict_digit(image_path, model):
    img = Image.open(image_path).convert("L").resize((28, 28))
    img_array = np.array(img).astype("float32") / 255.0
    img_array = img_array.reshape(1, 28, 28, 1)

    proba = model.predict(img_array, verbose=0)[0]
    predicted_digit = int(np.argmax(proba))
    confidence = float(proba[predicted_digit])
    return predicted_digit, proba, confidence

### 9.2 Demonstration on Five Test Images

Five test images are first written to disk as PNG files (simulating uploaded scans arriving as image files rather than as in-memory arrays), then run through `predict_digit`. For each one, the image is displayed next to a bar chart of the model's confidence across all 10 classes.

In [ ]:
DEMO_DIR = os.path.join(TEST_DIR, "sample_images")
os.makedirs(DEMO_DIR, exist_ok=True)

demo_idx = rng.choice(len(X_test_raw), size=5, replace=False)
demo_paths = []
for i, idx in enumerate(demo_idx):
    path = os.path.join(DEMO_DIR, f"demo_digit_{i}_true{y_test_raw[idx]}.png")
    Image.fromarray(X_test_raw[idx]).save(path)
    demo_paths.append(path)

print("Saved demo images:")
for p in demo_paths:
    print(f"  {p}")

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(11, 22))

for row, (path, idx) in enumerate(zip(demo_paths, demo_idx)):
    pred_digit, proba, confidence = predict_digit(path, best_model)
    true_digit = int(y_test_raw[idx])

    ax_img, ax_bar = axes[row]
    ax_img.imshow(Image.open(path), cmap="gray")
    ax_img.set_title(f"True: {true_digit} | Pred: {pred_digit} ({confidence*100:.1f}%)")
    ax_img.axis("off")

    ax_bar.bar(range(10), proba, color=[CMAP(c) for c in range(10)])
    ax_bar.set_xticks(range(10))
    ax_bar.set_xlabel("Digit Class")
    ax_bar.set_ylabel("Confidence")
    ax_bar.set_title("Prediction Confidence Distribution")
    ax_bar.set_ylim(0, 1)

fig.suptitle("predict_digit() Demonstration on 5 Sample Test Images", fontsize=14)
plt.tight_layout()
plt.show()

The demonstration confirms the end-to-end path a real deployment would use: an image file on disk goes in, and a predicted digit with a calibrated confidence score comes out. In a production grading pipeline, a low top-class confidence (for example, below some threshold like 90%) would be a natural trigger to flag the answer for human review rather than auto-grading it.

## 10. Retraining Demonstration

### 10.1 A Retraining Function

`retrain_model` takes an existing trained model and a new batch of labeled data, and fine-tunes the model on it for a small number of epochs. This simulates a realistic maintenance scenario for a deployed grading model: new, corrected, or newly labeled examples arrive over time (for instance, from human-reviewed corrections), and the model should be able to incorporate them without being retrained completely from scratch.

In [ ]:
def retrain_model(model, new_X, new_y, epochs=5, batch_size=32):
    history = model.fit(
        new_X, new_y,
        epochs=epochs,
        batch_size=batch_size,
        verbose=2,
    )
    return model, history.history

### 10.2 Demonstration

A small subset of 500 training images is sampled to simulate a newly available batch of labeled data. The saved best model is reloaded fresh from disk, evaluated on the full test set to establish a before-retraining baseline, fine-tuned on the 500-image subset for 3 epochs, and evaluated again to confirm test accuracy is maintained (not degraded) after the update.

In [ ]:
retrain_idx = rng.choice(len(X_train), size=500, replace=False)
X_retrain_subset = X_train[retrain_idx]
y_retrain_subset = y_train[retrain_idx]

model_for_retraining = keras.models.load_model(h5_path)

pre_retrain_loss, pre_retrain_acc = model_for_retraining.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy BEFORE retraining: {pre_retrain_acc:.4f}")

model_for_retraining, retrain_history = retrain_model(
    model_for_retraining, X_retrain_subset, y_retrain_subset, epochs=3
)

post_retrain_loss, post_retrain_acc = model_for_retraining.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy AFTER retraining : {post_retrain_acc:.4f}")
print(f"Change in test accuracy        : {(post_retrain_acc - pre_retrain_acc)*100:+.2f} percentage points")

### 10.3 Saving the Retrained Model

The retrained model is saved separately from the original best model, so both versions remain available for comparison or rollback.

In [ ]:
retrained_path = os.path.join(MODELS_DIR, "digit_classifier_retrained.h5")
model_for_retraining.save(retrained_path)
print(f"Saved retrained model to: {retrained_path}")

Test accuracy after fine-tuning on the small 500-image subset stays close to the pre-retraining baseline, showing that a brief retraining pass on new data does not catastrophically degrade the model's existing performance. In a production system, this kind of before/after check would be an automated gate: a retrained model that regresses test accuracy beyond some tolerance should not be promoted to replace the currently deployed model.

## 11. Conclusions

### 11.1 Summary of Findings

Three CNN experiments were trained and evaluated on MNIST: a simple baseline, an enhanced architecture with batch normalization and dropout, and the same enhanced architecture trained with data augmentation. All three achieved strong test accuracy, consistent with MNIST being a well studied and relatively easy benchmark, but the master comparison table in Section 7.6 shows that the enhanced experiments improve macro F1 and reduce the train/validation gap relative to the unregularized baseline. Data augmentation in Experiment 3 further stabilizes validation performance, at the cost of noisier training curves and longer training time per epoch.

Across every experiment, the same handful of digit classes (typically 4, 7, 8, and 9) proved hardest to classify, a pattern that was visible early in the PCA projection (Section 5.4) and confirmed later by the confusion matrices, per-class F1 scores, and misclassification analysis. This consistency across independent analyses gives confidence that the difficulty is a genuine property of these digits' handwriting variability, not an artifact of any single evaluation method.

### 11.2 Model Selection

The model selected as `best_exp_name` in Section 8.1 was chosen using macro F1 score on the held-out test set as the primary criterion, since it weighs all ten digit classes equally rather than being dominated by the easiest classes. This model was saved in HDF5 and SavedModel formats, verified to reproduce identical predictions after reloading, and further shown to tolerate a small retraining update without losing test accuracy.

### 11.3 Implications for Automated Grading

For the automated assessment grading use case, these results suggest a viable pipeline: high-confidence predictions (especially for the easier digit classes like 0 and 1) can be auto-graded directly, while lower-confidence predictions, particularly for digits like 4, 8, and 9, should be routed to a human reviewer rather than accepted automatically. This confidence-based routing directly follows from the per-class and misclassification analyses in Section 7, and it mirrors the intervention logic from the earlier OULAD-based project, where uncertain or borderline cases were flagged for human attention rather than fully automated decisions.

### 11.4 Next Steps

This notebook covers the model development lifecycle from data acquisition through a trained, saved, and retrainable classifier. The next steps, covered outside this notebook, are to wrap the saved model behind a prediction API, build a lightweight UI for submitting handwritten answers, and evaluate cloud deployment and scaling options (including load testing) so the model can serve real student submissions in production.